# Boids Separation Influence Field

A JAX flock simulation where separation is driven by a sampled influence field.

In [13]:
import time

import jax
import jax.numpy as jnp
from IPython.display import display
from ipycanvas import Canvas, hold_canvas

from flock_sim import (
    initialize_flock,
    separation_field,
    simulation_to_canvas,
    update_flock_with_separation_field,
)

In [14]:
population = 200
simulation_grid_size = 128
render_grid_size = 600

dt = 0.016
min_velocity = 0.05
max_velocity = 1.0

separation_strength = 1.0
turn_rate = 0.2
sigma = 0.08

In [15]:
rng_key = jax.random.key(10)
flock = initialize_flock(rng_key, population)

update_step = jax.jit(
    lambda flock: update_flock_with_separation_field(
        flock=flock,
        simulation_grid_size=simulation_grid_size,
        dt=dt,
        min_velocity=min_velocity,
        max_velocity=max_velocity,
        separation_strength=separation_strength,
        turn_rate=turn_rate,
        sigma=sigma,
    )
)

In [16]:
field_canvas = Canvas(width=render_grid_size, height=render_grid_size)
bird_canvas = Canvas(width=render_grid_size, height=render_grid_size)
display(field_canvas, bird_canvas)


def draw_field_height_map(canvas, field):
    magnitude = jnp.linalg.norm(field, axis=-1)
    magnitude = magnitude / jnp.maximum(jnp.max(magnitude), 1e-8)

    rows = jnp.linspace(0, simulation_grid_size - 1, render_grid_size).astype(jnp.int32)
    cols = jnp.linspace(0, simulation_grid_size - 1, render_grid_size).astype(jnp.int32)
    height_map = magnitude[rows[:, None], cols[None, :]]

    red = (25 + 40 * height_map).astype(jnp.uint8)
    green = (35 + 135 * height_map).astype(jnp.uint8)
    blue = (50 + 205 * height_map).astype(jnp.uint8)
    alpha = jnp.full_like(red, 255, dtype=jnp.uint8)
    image = jnp.stack([red, green, blue, alpha], axis=-1)
    canvas.put_image_data(jax.device_get(image), 0, 0)


def draw_birds(canvas, flock):
    points = simulation_to_canvas(flock.positions, render_grid_size)
    points = jax.device_get(points)
    canvas.clear()
    canvas.fill_style = "rgb(8, 10, 14)"
    canvas.fill_rect(0, 0, render_grid_size, render_grid_size)
    canvas.fill_style = "rgba(0, 0, 0, 0.8)"
    for x, y in points:
        canvas.fill_circle(float(x), float(y), 4.0)
    canvas.fill_style = "white"
    for x, y in points:
        canvas.fill_circle(float(x), float(y), 2.4)


def draw_initial_frame(flock):
    field = separation_field(flock, simulation_grid_size, sigma, separation_strength)
    with hold_canvas(field_canvas):
        field_canvas.clear()
        draw_field_height_map(field_canvas, field)
    with hold_canvas(bird_canvas):
        draw_birds(bird_canvas, flock)

Canvas(height=600, width=600)

Canvas(height=600, width=600)

In [17]:
draw_initial_frame(flock)

In [18]:
steps = 300

for step in range(steps):
    flock = update_step(flock)
    with hold_canvas(bird_canvas):
        draw_birds(bird_canvas, flock)
    time.sleep(dt)